# 05.01 Training Data Setup, Stratified 10-Fold CV Partitioning & Data Split Visualizations

**Phase 05 — Model Selection and Training (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Establish leak-proof cross-validation splits, validate feature schema alignment against `final_feature_schema.json`, verify strict target isolation, configure Stratified 10-Fold partitioning ($K=10$) on the development training partition ($N=350$), quarantine the final test set ($N=75$), and generate comprehensive academic visualizations of overall, partition-level, and fold-level class distributions.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Ingest `selected_features.csv` ($N=500$, 48 predictors) and align strictly with `final_feature_schema.json`.
2. Confirm absence of data leakage: exclude identifiers (`internal_record_id`), partition columns (`split_partition`), and post-decision target indicators.
3. Formulate binary target: `eligibility` (0 = `INELIGIBLE`, 1 = `ELIGIBLE`), balanced 50/50 in development training ($N=350$: 175/175).
4. Verify that each target class contains $\ge 10$ instances to rigorously satisfy $K=10$ Stratified Cross-Validation.
5. Establish reproducible Stratified 10-Fold cross-validation (`random_state=42`) on the training partition ($N=350$, exactly 35 samples per fold: 17 or 18 per class per fold).
6. Persist fold partition mapping to `data/processed/splits/cv_split_indices.json` and verify quarantine status of final test partition ($N=75$).
7. Produce publication-ready visualization graphs (PNG at 300 DPI and PDF) illustrating partition distributions, sample counts, fold heatmaps, and class balance.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary engineered feature dataset, 500 records x 51 columns)
- `data/processed/features/final_feature_schema.json` (Validated feature contract, 48 features)

**Expected Outputs:**
- `data/processed/splits/cv_split_indices.json` (Stratified 10-Fold train/val index mapping for models)
- `reports/phase_05/visualizations/training_data_setup/target_class_distribution_overall.{png,pdf}`
- `reports/phase_05/visualizations/training_data_setup/partition_sample_and_class_distribution.{png,pdf}`
- `reports/phase_05/visualizations/training_data_setup/stratified_10fold_heatmap.{png,pdf}`
- `reports/phase_05/visualizations/training_data_setup/stratified_10fold_proportions.{png,pdf}`
- `reports/phase_05/visualizations/training_data_setup/data_setup_summary_table.csv`

### 1. Environment Setup & Library Imports

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8

print(f"Libraries loaded. RANDOM_SEED={RANDOM_SEED}, Matplotlib v{plt.matplotlib.__version__}, Seaborn v{sns.__version__}")


### 2. Setup Paths & Load Selected Features and Contract Schema

In [ ]:
PROJECT_ROOT = Path("../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
SCHEMA_PATH = PROJECT_ROOT / "data" / "processed" / "features" / "final_feature_schema.json"
SPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
REPORT_VIZ_DIR = PROJECT_ROOT / "reports" / "phase_05" / "visualizations" / "training_data_setup"

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_VIZ_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert SCHEMA_PATH.exists(), f"Missing feature schema contract: {SCHEMA_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

print(f"Loaded selected_features.csv: {df_features.shape}")
print(f"Loaded feature schema version: {feature_schema.get('schema_version')}")


### 3. Validate Feature Contract & Verify Class Counts for K=10 Stratification

In [ ]:
# Extract 48 features from schema
contract_feature_names = [f["feature_name"] for f in feature_schema["features"]]
assert len(contract_feature_names) == 48, f"Expected 48 features, got {len(contract_feature_names)}"

missing_contract_feats = [col for col in contract_feature_names if col not in df_features.columns]
assert len(missing_contract_feats) == 0, f"Missing contract features in CSV: {missing_contract_feats}"

target_col = feature_schema["target_specification"]["column_name"]
assert target_col == "eligibility", f"Unexpected target column name: {target_col}"
assert target_col in df_features.columns, f"Target column {target_col} missing from dataset"

partition_counts = df_features["split_partition"].value_counts().to_dict()
print(f"Partition distribution: {partition_counts}")
assert partition_counts["train"] == 350, f"Expected 350 train samples, got {partition_counts.get('train')}"
assert partition_counts["validation"] == 75, f"Expected 75 val samples, got {partition_counts.get('validation')}"
assert partition_counts["test"] == 75, f"Expected 75 test samples, got {partition_counts.get('test')}"

# K=10 Precondition Check: Every class in train partition must have >= 10 samples
train_classes = df_features.loc[df_features["split_partition"] == "train", "eligibility"].value_counts().to_dict()
print(f"Training class sample counts: {train_classes}")
for cls_name, cls_cnt in train_classes.items():
    if cls_cnt < 10:
        raise ValueError(f"Class '{cls_name}' has {cls_cnt} instances (<10). Insufficient for 10-fold CV.")
print("Validation Check: Each target class has >= 10 instances (175 each). Stratified K=10 is fully supported.")


### 4. Configure Stratified 10-Fold Cross-Validation on Development Set

In [ ]:
# Isolate development training partition (N=350)
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_dev = df_train[contract_feature_names]
y_train_dev = (df_train["eligibility"] == "ELIGIBLE").astype(int)

CV_CONFIG = {
    "n_splits": 10,
    "shuffle": True,
    "random_state": RANDOM_SEED
}

skf = StratifiedKFold(
    n_splits=CV_CONFIG["n_splits"],
    shuffle=CV_CONFIG["shuffle"],
    random_state=CV_CONFIG["random_state"]
)

fold_splits = []
fold_summary_rows = []
for fold_idx, (t_idx, v_idx) in enumerate(skf.split(X_train_dev, y_train_dev)):
    val_elig = int(y_train_dev.iloc[v_idx].sum())
    val_inelig = int((1 - y_train_dev.iloc[v_idx]).sum())
    val_total = len(v_idx)
    elig_prop = val_elig / val_total
    
    fold_splits.append({
        "fold": fold_idx + 1,
        "train_indices": t_idx.tolist(),
        "val_indices": v_idx.tolist(),
        "train_samples": len(t_idx),
        "val_samples": val_total,
        "val_eligible": val_elig,
        "val_ineligible": val_inelig,
        "val_eligible_proportion": round(elig_prop, 4)
    })
    fold_summary_rows.append({
        "Fold": f"Fold {fold_idx + 1}",
        "Train Count": len(t_idx),
        "Val Count": val_total,
        "Eligible": val_elig,
        "Ineligible": val_inelig,
        "Eligible %": f"{elig_prop * 100:.1f}%"
    })

df_fold_summary = pd.DataFrame(fold_summary_rows)
print("=== STRATIFIED 10-FOLD CV SUMMARY (N=350) ===")
print(df_fold_summary.to_string(index=False))


### 5. Quarantine Final Test Set and Persist 10-Fold CV Configuration

In [ ]:
# Test set quarantine verification
test_mask = df_features["split_partition"] == "test"
df_test = df_features.loc[test_mask].reset_index(drop=True)
print(f"Quarantined Final Test Set verified: {df_test.shape} (N={len(df_test)} samples)")
print(f"Test partition class distribution: {df_test['eligibility'].value_counts().to_dict()}")
print("TEST ISOLATION ENFORCED: Test partition will remain strictly unread until final champion evaluation.")

cv_manifest = {
    "protocol": "StratifiedKFold-10Fold",
    "random_seed": RANDOM_SEED,
    "n_splits": 10,
    "total_dev_samples": len(df_train),
    "feature_count": len(contract_feature_names),
    "feature_columns": contract_feature_names,
    "target": "eligibility",
    "target_classes": ["INELIGIBLE (0)", "ELIGIBLE (1)"],
    "folds": fold_splits
}

cv_output_path = SPLITS_DIR / "cv_split_indices.json"
with open(cv_output_path, "w", encoding="utf-8") as f:
    json.dump(cv_manifest, f, indent=2)
print(f"Stratified 10-Fold CV configuration successfully saved to: {cv_output_path}")


### 6. Training, Validation, and Test Partition Visualizations

The following figures illustrate the verified project data partitioning:
- **Figure 1:** Overall Target Class Distribution ($N=500$).
- **Figure 2:** Train vs Validation vs Test Sample Counts and Class Balance.
- **Figure 3:** Stratified 10-Fold Class Distribution Heatmap across Folds.
- **Figure 4:** Stratified 10-Fold Class Proportion Comparison.

In [ ]:
# --- FIGURE 1: Overall Target Class Distribution ---
fig, ax = plt.subplots(figsize=(7, 5))
target_counts = df_features["eligibility"].value_counts()
colors = ["#2b5c8f", "#d95f02"]
bars = ax.bar(target_counts.index, target_counts.values, color=colors, width=0.45, edgecolor="black", linewidth=1.1)

ax.set_title("Overall Target Class Distribution (N=500)\nOFPHIL Charity Eligibility", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Number of Records", fontsize=11, fontweight="bold")
ax.set_xlabel("Eligibility Classification", fontsize=11, fontweight="bold")
ax.set_ylim(0, max(target_counts.values) * 1.25)

for bar in bars:
    h = bar.get_height()
    pct = (h / len(df_features)) * 100
    ax.annotate(f"{h} records\n({pct:.1f}%)",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 6), textcoords="offset points",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "target_class_distribution_overall.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "target_class_distribution_overall.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 1: target_class_distribution_overall (PNG/PDF)")


In [ ]:
# --- FIGURE 2: Partition Sample Counts and Class Breakdown ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Partition Counts
part_counts = df_features["split_partition"].value_counts()[["train", "validation", "test"]]
part_labels = ["Train (Dev)", "Validation", "Quarantined Test"]
part_colors = ["#1b9e77", "#7570b3", "#e7298a"]
bars1 = ax1.bar(part_labels, part_counts.values, color=part_colors, width=0.5, edgecolor="black", linewidth=1.1)
ax1.set_title("Partition Sample Count Breakdown (N=500)", fontsize=12, fontweight="bold", pad=10)
ax1.set_ylabel("Number of Records", fontsize=11, fontweight="bold")
ax1.set_ylim(0, 420)
for bar in bars1:
    h = bar.get_height()
    pct = (h / len(df_features)) * 100
    ax1.annotate(f"{h}\n({pct:.1f}%)", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Class counts by partition
part_class = pd.crosstab(df_features["split_partition"], df_features["eligibility"]).reindex(["train", "validation", "test"])
part_class.index = part_labels
part_class.plot(kind="bar", stacked=False, ax=ax2, color=["#2b5c8f", "#d95f02"], width=0.6, edgecolor="black", linewidth=1.1)
ax2.set_title("Class Distribution across Partitions", fontsize=12, fontweight="bold", pad=10)
ax2.set_ylabel("Record Count", fontsize=11, fontweight="bold")
ax2.set_xlabel("Partition", fontsize=11, fontweight="bold")
ax2.set_xticklabels(part_labels, rotation=0)
ax2.set_ylim(0, 210)
ax2.legend(title="Eligibility Class", frameon=True)
for p in ax2.patches:
    h = p.get_height()
    if h > 0:
        ax2.annotate(f"{int(h)}", xy=(p.get_x() + p.get_width() / 2, h),
                     xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "partition_sample_and_class_distribution.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "partition_sample_and_class_distribution.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 2: partition_sample_and_class_distribution (PNG/PDF)")


In [ ]:
# --- FIGURE 3 & 4: Stratified 10-Fold Heatmap & Class Proportions ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

fold_data = []
for f in fold_splits:
    fold_data.append({
        "Fold": f"Fold {f['fold']}",
        "INELIGIBLE (0)": f["val_ineligible"],
        "ELIGIBLE (1)": f["val_eligible"],
        "Total Val": f["val_samples"]
    })
df_folds_plot = pd.DataFrame(fold_data).set_index("Fold")

# Heatmap
sns.heatmap(df_folds_plot[["INELIGIBLE (0)", "ELIGIBLE (1)"]], annot=True, fmt="d", cmap="Blues",
            cbar=False, ax=ax1, linewidths=1.2, linecolor="white", annot_kws={"fontsize": 11, "fontweight": "bold"})
ax1.set_title("Stratified 10-Fold Class Distribution (Validation Folds)", fontsize=12, fontweight="bold", pad=12)
ax1.set_ylabel("Cross-Validation Fold", fontsize=11, fontweight="bold")

# Fold Proportion Comparison
fold_props = [f["val_eligible_proportion"] * 100 for f in fold_splits]
fold_names = [f"F{f['fold']}" for f in fold_splits]
bars = ax2.bar(fold_names, fold_props, color="#386cb0", width=0.55, edgecolor="black", linewidth=1.1)
ax2.axhline(50.0, color="red", linestyle="--", linewidth=1.5, label="Exact 50.0% Parity Line")
ax2.set_title("Eligible Class Proportion per Fold (%)", fontsize=12, fontweight="bold", pad=12)
ax2.set_ylabel("Eligible % in Fold Validation Set", fontsize=11, fontweight="bold")
ax2.set_xlabel("Fold Identifier", fontsize=11, fontweight="bold")
ax2.set_ylim(40, 60)
ax2.legend(loc="upper right", frameon=True)
for bar in bars:
    h = bar.get_height()
    ax2.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "stratified_10fold_heatmap.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "stratified_10fold_heatmap.pdf", bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "stratified_10fold_proportions.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "stratified_10fold_proportions.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figures 3 & 4: stratified_10fold_heatmap and proportions (PNG/PDF)")


### 7. Export Data Setup Summary Table and Manifests

In [ ]:
summary_table_rows = [
    {"Metric / Dimension": "Total Records Available", "Value": str(len(df_features))},
    {"Metric / Dimension": "Training Development Records (N_train)", "Value": f"{len(df_train)} (70.0%)"},
    {"Metric / Dimension": "Hold-out Validation Records (N_val)", "Value": f"{partition_counts['validation']} (15.0%)"},
    {"Metric / Dimension": "Quarantined Final Test Records (N_test)", "Value": f"{partition_counts['test']} (15.0%)"},
    {"Metric / Dimension": "CV Protocol", "Value": "StratifiedKFold (K=10, shuffle=True, seed=42)"},
    {"Metric / Dimension": "Number of Folds", "Value": "10"},
    {"Metric / Dimension": "Validation Samples per Fold", "Value": "35"},
    {"Metric / Dimension": "Target Variable", "Value": "eligibility (0: INELIGIBLE, 1: ELIGIBLE)"},
    {"Metric / Dimension": "Features Available", "Value": "48 (0 nulls, verified contract)"}
]
df_setup_manifest = pd.DataFrame(summary_table_rows)
df_setup_manifest.to_csv(REPORT_VIZ_DIR / "data_setup_summary_table.csv", index=False)
print(f"Data setup summary table exported: {REPORT_VIZ_DIR / 'data_setup_summary_table.csv'}")


### 8. Interpretation Guide for Researcher

- **Target Balance Verification:** Figure 1 confirms 50/50 balance ($250/250$) across the full dataset.
- **Partition Discipline:** Figure 2 confirms exact sample allocations ($350$ train, $75$ val, $75$ quarantined test) without overlap.
- **Stratified 10-Fold Parity:** Figures 3 and 4 verify that every validation fold contains exactly 35 records with balanced class representation (17 or 18 per class, approximately 48.6% to 51.4%), preventing fold-induced bias during hyperparameter tuning.

### Completion Checklist

- [ ] Target class balance verified across entire dataset and partitions.
- [ ] Precondition verified: $\ge 10$ samples per class in training partition.
- [ ] Stratified 10-Fold CV configured ($K=10$, `random_state=42`).
- [ ] Fold indices persisted to `cv_split_indices.json`.
- [ ] Test set verified strictly quarantined.
- [ ] Matplotlib and Seaborn graphs export to PNG (300 DPI) and PDF.
- [ ] Summary table exported to CSV.